# Remote Cleanup 01: Baseline and Individual Release

This notebook mirrors the first manual pytest stages:

1. Run the normal remote overlay/DMA loopback.
2. Create remote MMIO, buffer, and optional GPIO objects.
3. Pause so you can inspect the board logs.
4. Delete the Python objects individually and verify that the server-side IDs are gone.

Set `USE_PROBED_DEVICE` to `True` if you want to use the probed remote device discovered through `Device.devices`. Leave it `False` if you want to construct `RemoteDevice(...)` manually.

In [ ]:
import gc
import os
from pathlib import Path

import grpc
import numpy as np

REMOTE_IP = os.environ.get("PYNQ_REMOTE_DEVICES", "192.168.2.197").split(",")[0].strip()
USE_PROBED_DEVICE = False
OVERLAY_NAME = "resizer.xsa"

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "tests" / OVERLAY_NAME).exists():
            return candidate
    raise FileNotFoundError(f"Could not find tests/{OVERLAY_NAME} from {start}")

REPO_ROOT = find_repo_root(Path.cwd())
OVERLAY_PATH = REPO_ROOT / "tests" / OVERLAY_NAME

os.environ["PYNQ_REMOTE_DEVICES"] = REMOTE_IP

from pynq import GPIO, Overlay
from pynq.pl_server.device import Device
from pynq.pl_server.remote_device import RemoteDevice
from pynq.remote import buffer_pb2, gpio_pb2, mmio_pb2

def get_device(auto_cleanup: bool = True):
    if USE_PROBED_DEVICE:
        devices = [d for d in Device.devices if isinstance(d, RemoteDevice)]
        if not devices:
            raise RuntimeError("No probed RemoteDevice found. Check PYNQ_REMOTE_DEVICES and connectivity.")
        if auto_cleanup is False:
            print("Using the probed device instance. Its auto_cleanup policy was chosen when it was constructed.")
        return devices[0]
    return RemoteDevice(ip_addr=REMOTE_IP, auto_cleanup=auto_cleanup)

print(f"REMOTE_IP={REMOTE_IP}")
print(f"USE_PROBED_DEVICE={USE_PROBED_DEVICE}")
print(f"OVERLAY_PATH={OVERLAY_PATH}")

In [ ]:
device = get_device(auto_cleanup=True)
overlay = Overlay(str(OVERLAY_PATH), device=device)

dma = overlay.axi_dma_0
resizer = overlay.resize_accel_0
size = 500
fake_img = np.random.randint(0, 256, (size, size, 3), dtype=np.uint8)

in_buffer = device.allocate(shape=(size, size, 3), dtype=np.uint8, cacheable=1)
out_buffer = device.allocate(shape=(size, size, 3), dtype=np.uint8, cacheable=1)
in_buffer[:] = fake_img

resizer.register_map.src_rows = size
resizer.register_map.src_cols = size
resizer.register_map.dst_rows = size
resizer.register_map.dst_cols = size

dma.sendchannel.transfer(in_buffer)
dma.recvchannel.transfer(out_buffer)
resizer.write(0x00, 0x81)
dma.sendchannel.wait()
dma.recvchannel.wait()

assert np.array_equal(in_buffer[:], out_buffer[:])
print("Baseline remote loopback passed.")

## Individual object release

The next cell creates a manual remote MMIO object, a manual remote buffer, and an optional remote GPIO object. It then pauses so you can inspect the board logs before deleting the Python objects individually.

In [ ]:
base_addr = overlay.ip_dict["resize_accel_0"]["phys_addr"]

remote_mmio = device.mmap(base_addr, 0x1000)
mmio_id = remote_mmio.mmio_id
remote_mmio.read(0)

remote_buffer = device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
buffer_id = remote_buffer.buffer_id
remote_buffer[:] = np.arange(16, dtype=np.uint32)
remote_buffer.flush()

remote_gpio = None
gpio_id = None
gpio_path = None
base_path = GPIO.get_gpio_base_path(device=device)
npins = GPIO.get_gpio_npins(device=device)
if base_path and npins:
    gpio_pin = GPIO.get_gpio_pin(0, device=device)
    gpio_path = f"/sys/class/gpio/gpio{gpio_pin}"
    if not device.exists_file(gpio_path).exists:
        remote_gpio = GPIO(gpio_pin, "in", device=device)
        remote_gpio.read()
        gpio_id = remote_gpio._gpio_id
    else:
        print(f"Skipping GPIO because {gpio_path} is already exported.")
else:
    print("Skipping GPIO because Linux sysfs GPIO is not available.")

print(f"MMIO id:   {mmio_id}")
print(f"Buffer id: {buffer_id}")
if gpio_id is not None:
    print(f"GPIO id:   {gpio_id} at {gpio_path}")

input("Check the board logs for object creation, then press Enter to delete these Python objects individually.")

del remote_mmio
del remote_buffer
del remote_gpio
gc.collect()

input("Check the board logs for release_mmio/freebuffer/unexport, then press Enter to verify the server-side IDs are gone.")

In [ ]:
try:
    device._stub["mmio"].read(
        mmio_pb2.ReadRequest(mmio_id=mmio_id, offset=0, length=4, word_order="little")
    )
    raise AssertionError(f"MMIO id {mmio_id} still exists on the server.")
except (grpc.RpcError, RuntimeError):
    print(f"MMIO id {mmio_id} is invalid as expected.")

try:
    device._stub["buffer"].physical_address(buffer_pb2.AddressRequest(buffer_id=buffer_id))
    raise AssertionError(f"Buffer id {buffer_id} still exists on the server.")
except (grpc.RpcError, RuntimeError):
    print(f"Buffer id {buffer_id} is invalid as expected.")

if gpio_id is not None:
    try:
        device._stub["gpio"].read(gpio_pb2.GpioReadRequest(gpio_id=gpio_id))
        raise AssertionError(f"GPIO id {gpio_id} still exists on the server.")
    except (grpc.RpcError, RuntimeError):
        print(f"GPIO id {gpio_id} is invalid as expected.")
    assert device.exists_file(gpio_path).exists is False
    print(f"GPIO sysfs path {gpio_path} has been removed.")